# Phase 2 Mesh — Run Record


In [ ]:
import yaml
from pathlib import Path

config_path = Path("../../config/mesh.yaml")
print(yaml.safe_load(config_path.read_text()))


In [ ]:
import asyncio
import sys
import nest_asyncio
nest_asyncio.apply()  # Required: Jupyter already has a running event loop

sys.path.insert(0, str(Path("../../src")))
sys.path.insert(0, str(Path("../../")))

from aip_intern.core.config import load_config
from aip_intern.baseline.runner import RunConfig
from aip_intern.mesh.runner import run

app_cfg = load_config(config_path)
run_cfg = RunConfig(
    run_id_prefix=app_cfg.run.run_id_prefix,
    n_runs=app_cfg.run.n_runs,
    config_path=config_path,
    llm_model=app_cfg.llm.model,
    llm_base_url=app_cfg.llm.base_url,
    llm_api_key=app_cfg.llm.api_key,
    workspace_root=Path("../../workspace/"),
    artifacts_dir=Path("../../artifacts/"),
)

# TODO: execute before final commit — requires OPENAI_BASE_URL and OPENAI_API_KEY to be set
results = asyncio.run(run(run_cfg))
print(f"\n{sum(r.success for r in results)}/{len(results)} runs succeeded")


In [ ]:
import pandas as pd

last_success = next((r for r in reversed(results) if r.success), None)
if last_success:
    triage_path = last_success.outputs_path / "triage.csv"
    if triage_path.exists():
        display(pd.read_csv(triage_path))


In [ ]:
from IPython.display import Markdown
if last_success:
    brief_path = last_success.outputs_path / "brief.md"
    if brief_path.exists():
        display(Markdown(brief_path.read_text()))


In [ ]:
import json
if last_success:
    metrics_path = last_success.outputs_path.parent / "metrics.json"
    if metrics_path.exists():
        print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
